# Maquina

In [ ]:
import json
import os
from datetime import datetime

# ── Identificação da máquina ──────────────────────────────────────────────────
# Mude isso em cada máquina antes de rodar
MACHINE_ID = "pc01"   # ex: "pc1", "pc2", "kaggle_01", "kaggle_02"

# Pasta onde os trials serão salvos
SAVE_DIR = f"trials_{MACHINE_ID}"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Machine ID : {MACHINE_ID}")
print(f"Save dir   : {SAVE_DIR}/")

In [ ]:
import shutil
import glob

# Copia os arquivos do dataset de input para o SAVE_DIR
src = "/kaggle/input/datasets/jniorviana/pc01-36trials/trials_pc01"   # ← caminho do dataset que você subiu
os.makedirs(SAVE_DIR, exist_ok=True)

for f in glob.glob(os.path.join(src, "*")):
    shutil.copy(f, SAVE_DIR)
    print(f"Copiado: {f}")

print(f"\nArquivos em {SAVE_DIR}:")
print(os.listdir(SAVE_DIR))

# Dataset

In [ ]:
import os
import copy
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import optuna

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Dataset

In [ ]:
# import os
# import numpy as np
# from PIL import Image
# from torch.utils.data import Dataset, DataLoader
# from torchvision import datasets

# DATASET_DIR = "/kaggle/input/datasets/arnabkumarroy02/ferplus"

# # Carrega as 3 pastas e junta tudo num pool único
# all_samples = []
# all_labels  = []

# for split in ["train", "validation", "test"]:
#     folder = datasets.ImageFolder(os.path.join(DATASET_DIR, split))
#     all_samples += folder.samples   # lista de (path, label)
#     all_labels  += folder.targets   # lista de ints

# all_samples = np.array(all_samples, dtype=object)
# all_labels  = np.array(all_labels,  dtype=int)

# classes     = folder.classes
# num_classes = len(classes)

# print(f"Total de imagens : {len(all_samples)}")
# print(f"Classes          : {classes}")
# print(f"Num classes      : {num_classes}")

In [ ]:
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

class FERPlusDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples   # lista de (path, label)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
DATASET_DIR = "/kaggle/input/datasets/arnabkumarroy02/ferplus"

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

transform_eval = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

# Carrega as 3 pastas e junta tudo num pool único
all_samples = []
all_labels  = []

for split in ["train", "validation", "test"]:
    folder = datasets.ImageFolder(os.path.join(DATASET_DIR, split))
    all_samples += folder.samples
    all_labels  += folder.targets

all_samples = np.array(all_samples, dtype=object)
all_labels  = np.array(all_labels,  dtype=int)

classes     = folder.classes
num_classes = len(classes)
BATCH_SIZE  = 64

print(f"Total de imagens : {len(all_samples)}")
print(f"Tipo all_labels  : {type(all_labels)} | dtype: {all_labels.dtype}")
print(f"Classes          : {classes}")
print(f"Num classes      : {num_classes}")


class FERPlusDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

In [ ]:
# from collections import Counter
# import matplotlib.pyplot as plt

# label_counts = Counter(all_labels.tolist() if isinstance(all_labels, np.ndarray) else all_labels)

# print("Distribuição do pool completo:")
# for i, cls in enumerate(classes):
#     print(f"  {cls:10s}: {label_counts[i]:6d} amostras")

# plt.figure(figsize=(8, 4))
# plt.bar(classes, [label_counts[i] for i in range(num_classes)])
# plt.title("Distribuição de classes — Pool completo")
# plt.xticks(rotation=30)
# plt.tight_layout()
# plt.show()

# Stem

In [ ]:
class Stem(nn.Module):
    def __init__(self, out_ch=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.stem(x)

# PlainConv

In [ ]:
class PlainConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        # Projeção 1×1 se canais ou stride mudam
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)

# MBConv

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, ratio=4):
        super().__init__()
        r = max(1, ch // ratio)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(r, ch, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.se(x).view(x.size(0), -1, 1, 1)
        return x * w


class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=6, kernel_size=3, stride=1):
        super().__init__()
        mid_ch  = in_ch * expand_ratio
        padding = kernel_size // 2

        self.conv = nn.Sequential(
            # Expand
            nn.Conv2d(in_ch, mid_ch, 1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            # Depthwise
            nn.Conv2d(mid_ch, mid_ch, kernel_size,
                      stride=stride, padding=padding,
                      groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            # SE
            SEBlock(mid_ch),
            # Project
            nn.Conv2d(mid_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )

        # Projeção 1×1 (opção A)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)

In [ ]:
class AnyNetStage(nn.Module):
    def __init__(self, in_ch, out_ch, depth, block_type):
        super().__init__()
        blocks = []
        for i in range(depth):
            stride = 2 if i == 0 else 1
            blocks.append(block_type(in_ch if i == 0 else out_ch, out_ch, stride=stride))
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)

In [ ]:
class AnyNet(nn.Module):
    def __init__(
        self,
        num_stages:       int,
        widths:           list,
        depths:           list,
        transition_stage: int,
        dropout:          float,
        num_classes:      int = 8,
    ):
        super().__init__()

        self.stem = Stem(out_ch=32)

        stages = []
        in_ch  = 32
        for i in range(num_stages):
            block_type = MBConv if i >= transition_stage - 1 else PlainConv
            stages.append(AnyNetStage(in_ch, widths[i], depths[i], block_type))
            in_ch = widths[i]
        self.stages = nn.Sequential(*stages)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(widths[-1], num_classes),
        )

        # Registra a camada alvo para Grad-CAM
        # Sempre o último bloco do último estágio, independente do tipo
        last_stage = list(self.stages.children())[-1]
        last_block = list(last_stage.blocks.children())[-1]
        self.gradcam_target = last_block.conv[-1]

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        return self.head(x)

# AnynetBuild

In [ ]:
def build_anynet(trial=None, config=None, num_classes=8) -> AnyNet:

    if trial is not None:
        num_stages       = trial.suggest_int("num_stages", 2, 4)
        transition_stage = trial.suggest_int("transition_stage", 1, num_stages + 1)
        dropout          = trial.suggest_float("dropout", 0.1, 0.5)

        widths = sorted([
            trial.suggest_int(f"width_s{i}", 16, 256, step=16)
            for i in range(num_stages)
        ])

        depths = [trial.suggest_int(f"depth_s{i}", 1, 4) for i in range(num_stages)]

    else:
        num_stages       = config["num_stages"]
        widths           = config["widths"]
        depths           = config["depths"]
        transition_stage = config["transition_stage"]
        dropout          = config["dropout"]

    return AnyNet(
        num_stages       = num_stages,
        widths           = widths,
        depths           = depths,
        transition_stage = transition_stage,
        dropout          = dropout,
        num_classes      = num_classes,
    )

# Gradcam

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import os
import random

# ── Grad-CAM ──────────────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model):
        self.model       = model
        self.activations = None
        self.gradients   = None

        # Usa a camada registrada no modelo — funciona para qualquer arquitetura
        self.target_layer = model.gradcam_target
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, _, __, output):
        self.activations = output.detach()

    def _save_gradient(self, _, __, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, img_tensor, class_idx):
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(device)

        self.model.zero_grad()
        logits = self.model(img_tensor)
        logits[0, class_idx].backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, size=(48, 48), mode="bilinear", align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def tensor_to_img(tensor):
    """Converte tensor normalizado de volta para imagem visualizável."""
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = img * 0.5 + 0.5   # desfaz Normalize(0.5, 0.5)
    return np.clip(img, 0, 1)


def overlay_cam(img_np, cam):
    """Sobrepõe o mapa de calor na imagem."""
    heatmap = plt.cm.jet(cam)[..., :3]
    overlay = 0.5 * img_np + 0.5 * heatmap
    return np.clip(overlay, 0, 1)


# ── Coleta predições ──────────────────────────────────────────────────────────
@torch.no_grad()
def collect_predictions(model, loader):
    """Retorna tensores de imagens, labels verdadeiros e predições."""
    model.eval()
    all_imgs, all_labels, all_preds = [], [], []
    for imgs, labels in loader:
        outputs = model(imgs.to(device))
        preds   = outputs.argmax(1).cpu()
        all_imgs.append(imgs)
        all_labels.append(labels)
        all_preds.append(preds)
    return (
        torch.cat(all_imgs),
        torch.cat(all_labels).numpy(),
        torch.cat(all_preds).numpy(),
    )


# ── Análise Grad-CAM ──────────────────────────────────────────────────────────
def gradcam_analysis(
    model,
    classes,
    loader      = None,
    image_dir   = None,
    target_class= None,
):
    """
    Gera figura Grad-CAM com VP, FP, FN, VN para uma classe.

    Parâmetros:
        model        : modelo treinado
        classes      : lista de nomes das classes
        loader       : DataLoader do fold de teste (modo dataset)
        image_dir    : path para pasta externa com subpastas por classe (modo externo)
        target_class : índice da classe alvo — None sorteia automaticamente
    """
    assert loader is not None or image_dir is not None, \
        "Forneça loader ou image_dir"

    # ── Carrega imagens ───────────────────────────────────────────────────
    if loader is not None:
        imgs, true_labels, pred_labels = collect_predictions(model, loader)

    else:
        # Modo externo: carrega imagens de image_dir/classe/
        imgs_list, labels_list = [], []
        for class_idx, class_name in enumerate(classes):
            class_path = os.path.join(image_dir, class_name)
            if not os.path.exists(class_path):
                continue
            files = [f for f in os.listdir(class_path)
                     if f.lower().endswith((".png", ".jpg", ".jpeg"))]
            for fname in files:
                img = Image.open(os.path.join(class_path, fname)).convert("RGB")
                img = transform_eval(img)
                imgs_list.append(img)
                labels_list.append(class_idx)

        imgs        = torch.stack(imgs_list)
        true_labels = np.array(labels_list)
        with torch.no_grad():
            model.eval()
            outputs     = model(imgs.to(device))
            pred_labels = outputs.argmax(1).cpu().numpy()

    # ── Sorteia classe alvo ───────────────────────────────────────────────
    if target_class is None:
        target_class = random.randint(0, len(classes) - 1)
    class_name = classes[target_class]
    print(f"Classe alvo: {class_name} (idx={target_class})")

    # ── Identifica VP, FP, FN, VN ─────────────────────────────────────────
    # Para a classe alvo no estilo one-vs-rest
    is_target_true = (true_labels == target_class)
    is_target_pred = (pred_labels == target_class)

    vp_idx = np.where( is_target_true &  is_target_pred)[0]
    fp_idx = np.where(~is_target_true &  is_target_pred)[0]
    fn_idx = np.where( is_target_true & ~is_target_pred)[0]
    vn_idx = np.where(~is_target_true & ~is_target_pred)[0]

    categories = {
        "VP (era, acertou)"   : vp_idx,
        "FP (não era, disse)" : fp_idx,
        "FN (era, não disse)" : fn_idx,
        "VN (não era, ok)"    : vn_idx,
    }

    for cat, idx in categories.items():
        print(f"  {cat}: {len(idx)} imagens disponíveis")

    # ── Grad-CAM ──────────────────────────────────────────────────────────
    gradcam = GradCAM(model)

    fig, axes = plt.subplots(
        nrows=4, ncols=2,
        figsize=(6, 12),
    )
    fig.suptitle(f"Grad-CAM — Classe: {class_name}", fontsize=14, fontweight="bold")

    colors = {
        "VP (era, acertou)"   : "green",
        "FP (não era, disse)" : "red",
        "FN (era, não disse)" : "orange",
        "VN (não era, ok)"    : "blue",
    }

    for row, (cat_name, idx_arr) in enumerate(categories.items()):
        for col in range(2):
            ax = axes[row, col]

            if len(idx_arr) == 0:
                ax.text(0.5, 0.5, "Sem exemplos", ha="center", va="center")
                ax.axis("off")
                continue

            # Pega sample aleatório da categoria
            sample_idx  = np.random.choice(idx_arr)
            img_tensor  = imgs[sample_idx]
            true_label  = true_labels[sample_idx]
            pred_label  = pred_labels[sample_idx]

            # Gera mapa Grad-CAM para a classe predita
            cam     = gradcam.generate(img_tensor, class_idx=int(pred_label))
            img_np  = tensor_to_img(img_tensor)
            overlay = overlay_cam(img_np, cam)

            ax.imshow(overlay)
            ax.set_title(
                f"Real: {classes[true_label]}\nPred: {classes[pred_label]}",
                fontsize=8,
                color=colors[cat_name],
            )
            ax.axis("off")

        # Label da categoria na linha
        axes[row, 0].set_ylabel(cat_name, fontsize=9, rotation=90, labelpad=40)

    plt.tight_layout()
    plt.savefig(f"gradcam_{class_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvo: gradcam_{class_name}.png")

# Treino e Validação

In [ ]:
from sklearn.metrics import f1_score as sk_f1, classification_report, confusion_matrix

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs      = model(imgs)
        loss         = criterion(outputs, labels)
        preds        = outputs.argmax(1)

        running_loss += loss.item() * imgs.size(0)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    avg_loss   = running_loss / total
    acc        = correct / total
    f1_macro   = sk_f1(all_labels, all_preds, average="macro", zero_division=0)

    if return_preds:
        return avg_loss, acc, f1_macro, all_preds, all_labels
    return avg_loss, acc, f1_macro

# Cross Validation

In [ ]:
import wandb
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score as sk_f1

N_FOLDS       = 3
VAL_SPLIT     = 0.10
EPOCHS_TRIAL  = 8
BATCH_SIZE    = 64
N_TRIALS      = 60
WANDB_PROJECT = "fer-anynet"

from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def objective(trial):
    num_stages       = trial.suggest_int("num_stages", 2, 4)
    transition_stage = trial.suggest_int("transition_stage", 1, num_stages + 1)
    dropout          = trial.suggest_float("dropout", 0.1, 0.5)
    widths           = sorted([
        trial.suggest_int(f"width_s{i}", 16, 256, step=16)
        for i in range(num_stages)
    ])
    depths = [trial.suggest_int(f"depth_s{i}", 1, 4) for i in range(num_stages)]
    lr     = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    run = wandb.init(
        project=WANDB_PROJECT,
        group="optuna_search",
        name=f"trial_{trial.number}_{MACHINE_ID}",
        config={
            "machine_id"      : MACHINE_ID,
            "trial"           : trial.number,
            "num_stages"      : num_stages,
            "widths"          : widths,
            "depths"          : depths,
            "transition_stage": transition_stage,
            "dropout"         : dropout,
            "lr"              : lr,
        },
        reinit=True,
    )

    fold_f1s = []

    for fold_idx, (trainval_idx, test_idx) in enumerate(
        skf.split(all_samples, all_labels), start=1
    ):
        train_idx, val_idx = train_test_split(
            trainval_idx,
            test_size=VAL_SPLIT,
            random_state=SEED,
            stratify=all_labels[trainval_idx],
        )

        train_loader_fold = DataLoader(
            FERPlusDataset(all_samples[train_idx].tolist(), transform=transform_train),
            batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True,
        )
        val_loader_fold = DataLoader(
            FERPlusDataset(all_samples[val_idx].tolist(), transform=transform_eval),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
        )

        model     = build_anynet(trial=trial, num_classes=num_classes).to(device)
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_TRIAL)

        best_val_f1 = 0.0
        for epoch in range(EPOCHS_TRIAL):
            train_loss, train_acc = train_one_epoch(
                model, train_loader_fold, criterion, optimizer, device
            )
            _, val_acc, val_f1 = evaluate(
                model, val_loader_fold, criterion, device
            )
            scheduler.step()

            wandb.log({
                "fold"      : fold_idx,
                "epoch"     : epoch + 1,
                "train_loss": train_loss,
                "train_acc" : train_acc,
                "val_acc"   : val_acc,
                "val_f1"    : val_f1,
            })

            best_val_f1 = max(best_val_f1, val_f1)

        fold_f1s.append(best_val_f1)
        del model
        torch.cuda.empty_cache()

    mean_f1 = float(np.mean(fold_f1s))
    std_f1  = float(np.std(fold_f1s))

    # ── Salva JSON do trial ───────────────────────────────────────────────
    trial_data = {
        "machine_id"  : MACHINE_ID,
        "trial_id"    : f"{MACHINE_ID}_trial_{trial.number}",
        "trial_number": trial.number,
        "mean_val_f1" : round(mean_f1, 6),
        "std_val_f1"  : round(std_f1,  6),
        "val_f1_folds": [round(float(f), 6) for f in fold_f1s],
        "config": {
            "num_stages"      : num_stages,
            "widths"          : widths,
            "depths"          : depths,
            "transition_stage": transition_stage,
            "dropout"         : round(float(dropout), 6),
        },
        "hparams": {
            "lr"          : round(float(lr), 8),
            "weight_decay": 1e-4,
        },
        "epochs_trial": EPOCHS_TRIAL,
        "n_folds"     : N_FOLDS,
        "dataset"     : "ferplus",
        "timestamp"   : datetime.now().isoformat(),
    }

    save_path = os.path.join(SAVE_DIR, f"{MACHINE_ID}_trial_{trial.number}.json")
    with open(save_path, "w") as f:
        json.dump(trial_data, f, indent=2)

    wandb.summary["mean_val_f1"] = mean_f1
    wandb.summary["std_val_f1"]  = std_f1
    wandb.summary["fold_f1s"]    = fold_f1s
    wandb.summary["trial_id"]    = f"{MACHINE_ID}_trial_{trial.number}"
    run.finish()

    print(f"  Trial {trial.number} — "
          f"F1 por fold: {[f'{f:.4f}' for f in fold_f1s]} | "
          f"Média: {mean_f1:.4f} ± {std_f1:.4f} | "
          f"Salvo: {save_path}")

    return mean_f1


# ── Study com persistência em SQLite ──────────────────────────────────────────
storage = optuna.storages.RDBStorage(
    url=f"sqlite:///{SAVE_DIR}/optuna_study_{MACHINE_ID}.db"
)

study = optuna.create_study(
    study_name=f"anynet_{MACHINE_ID}",
    direction="maximize",
    storage=storage,
    load_if_exists=True,   # retoma de onde parou se sessão cair
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

completed_trials = len([
    t for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
])
remaining_trials = max(0, N_TRIALS - completed_trials)

print(f"Trials já completos: {completed_trials}")
print(f"Trials restantes   : {remaining_trials}")
print(f"Trials no storage  : {len(study.trials)}")

if remaining_trials > 0:
    study.optimize(objective, n_trials=remaining_trials)
else:
    print("Todos os trials planejados já foram concluídos.")

completed_trials = len([
    t for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
])

if completed_trials > 0:
    print(f"\nMelhor trial  : {study.best_trial.number}")
    print(f"Melhor val_f1 : {study.best_value:.4f}")
    print(f"Params        : {study.best_params}")
else:
    print("\nNenhum trial completo ainda.")

# Inferencia Grad-Cam

In [ ]:
# # Modo dataset — usa o loader do fold
# gradcam_analysis(
#     model   = best_model,
#     classes = classes,
#     loader  = test_loader_fold,
# )

# # Modo externo — pasta com subpastas por classe
# gradcam_analysis(
#     model      = best_model,
#     classes    = classes,
#     image_dir  = "/caminho/outro/dataset",
#     target_class = 4,  # None para sortear
# )

# Search


In [ ]:
# # ─── Optuna Search ───────────────────────────────────────────────────────────

# def objective(trial):
#     model = build_anynet(trial=trial, num_classes=num_classes)
#     lr    = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
#     return fit(model, train_loader, val_loader, epochs=5, lr=lr, device=device)


# study = optuna.create_study(
#     direction="maximize",
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=2),  # mata trials ruins cedo
# )
# study.optimize(objective, n_trials=20)

# print("\nMelhor val_acc :", study.best_value)
# print("Melhores params:", study.best_params)

# Treino com a melhor config encontrada

In [ ]:
# # ─── Treino Final com Melhor Config ─────────────────────────────────────────

# best_model = build_anynet(config={
#     "num_stages"       : study.best_params["num_stages"],
#     "widths"           : sorted([study.best_params[f"width_s{i}"]
#                                  for i in range(study.best_params["num_stages"])]),
#     "depths"           : [study.best_params[f"depth_s{i}"]
#                           for i in range(study.best_params["num_stages"])],
#     "transition_stage" : study.best_params["transition_stage"],
#     "dropout"          : study.best_params["dropout"],
# }, num_classes=num_classes)

# fit(best_model, train_loader, val_loader, epochs=30,
#     lr=study.best_params["lr"], device=device)

# torch.save(best_model.state_dict(), "anynet_ferplus_best.pth")
# print("Modelo salvo.")

In [ ]:
# # avaliação final honesta
# criterion = nn.CrossEntropyLoss()
# test_loss, test_acc = evaluate(best_model, test_loader, criterion, device)
# print(f"Test acc final: {test_acc:.4f}")